In [8]:
# ================================================================
#        ARXIV RESEARCH PAPER CLASSIFIER — COMPLETE CODE
#              Using DistilBERT + Hugging Face Trainer
#
#  HOW TO USE THIS FILE:
#  1. Open Google Colab
#  2. Set Runtime → Change Runtime Type → GPU
#  3. Copy each CELL into a separate Colab cell
#  4. Run cells one by one from top to bottom
# ================================================================


# ================================================================
# CELL 1 — INSTALL ALL REQUIRED LIBRARIES
#
# WHY: Google Colab does not have kagglehub or transformers
# installed by default. We install them before importing anything.
# Run this cell first and wait for it to finish completely.
# ================================================================

# Uncomment and run this in Colab:
# !pip install kagglehub transformers datasets torch scikit-learn pandas -q


# ================================================================
# CELL 2 — IMPORT ALL LIBRARIES
#
# WHY: We bring in every tool we need at the top so nothing
# is missing later when we need it.
#
#   pandas        → handle data in table format
#   numpy         → numerical operations
#   re            → text cleaning using patterns
#   os            → file and folder operations
#   pickle        → save and load Python objects to disk
#   torch         → PyTorch deep learning framework
#   kagglehub     → download arXiv dataset from Kaggle
#   datasets      → Hugging Face dataset format (works with Trainer)
#   transformers  → DistilBERT model and tokenizer
#   sklearn       → label encoding and train/test split
# ================================================================

import pandas as pd
import numpy as np
import re
import os
import pickle
import torch

import kagglehub
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# Check GPU — must show 'cuda' for BERT to train at reasonable speed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# If device shows 'cpu', go to Runtime → Change Runtime Type → GPU
# and re-run from Cell 1



Using device: cuda


In [9]:
# ================================================================
# CELL 3 — DOWNLOAD THE ARXIV DATASET FROM KAGGLE
#
# WHY: We need a large collection of labelled research papers
# to train our classifier. The arXiv dataset from Cornell
# University has 1.7 million papers already sorted by category.
# kagglehub downloads it directly into the Colab environment.
#
# The dataset is about 5 GB so this step may take a few minutes.
# After downloading, we detect whether it is a CSV or JSON file
# and set data_file_path so all later cells can use it.
# chunk_size controls how many rows we read at a time — 100,000
# rows keeps RAM usage low while still processing efficiently.
# ================================================================

path  = kagglehub.dataset_download("Cornell-University/arxiv")
files = os.listdir(path)
print(f"Files downloaded: {files}")

chunk_size = 100_000   # process 100,000 rows at a time to save RAM

csv_files  = [f for f in files if f.endswith('.csv')]
json_files = [f for f in files if f.endswith('.json')]

if csv_files:
    data_file_path = os.path.join(path, csv_files[0])
    print(f"Using CSV file: {data_file_path}")
elif json_files:
    data_file_path = os.path.join(path, json_files[0])
    print(f"Using JSON file: {data_file_path}")
else:
    raise FileNotFoundError("No CSV or JSON file found in downloaded dataset.")

print(f"chunk_size: {chunk_size} rows per chunk")


Files downloaded: ['arxiv-metadata-oai-snapshot.json']
Using JSON file: C:\Users\dipak\.cache\kagglehub\datasets\Cornell-University\arxiv\versions\280\arxiv-metadata-oai-snapshot.json
chunk_size: 100000 rows per chunk


In [10]:
# ================================================================
# CELL 4 — DEFINE CATEGORY MAP AND HELPER FUNCTIONS
#
# WHY: The arXiv categories column contains codes like 'cs.AI'
# or 'math.ST'. We need to convert these codes into readable
# category names like 'Computer Science' or 'Mathematics'.
#
# category_map: dictionary that maps each prefix to a full name.
# There are 20 main categories covering all major academic fields.
#
# get_main_category: takes the full category string like 'cs.AI
# math.ST', picks only the FIRST category (since a paper belongs
# primarily to its first listed category), extracts the prefix
# before the dot, then looks it up in category_map. Returns None
# if the prefix is not in our map — those rows get dropped later.
#
# clean_text: removes HTML tags and collapses extra whitespace.
# We do LIGHT cleaning only because DistilBERT was pre-trained
# on natural English and understands punctuation and capitalisation.
# Removing them would hurt the model's understanding.
# ================================================================

category_map = {
    'cs'      : 'Computer Science',
    'math'    : 'Mathematics',
    'stat'    : 'Statistics',
    'astro-ph': 'Astrophysics',
    'hep-ph'  : 'High Energy Physics - Phenomenology',
    'hep-th'  : 'High Energy Physics - Theory',
    'quant-ph': 'Quantum Physics',
    'cond-mat': 'Condensed Matter',
    'nucl-ex' : 'Nuclear Experiment',
    'nucl-th' : 'Nuclear Theory',
    'gr-qc'   : 'General Relativity and Quantum Cosmology',
    'physics' : 'Physics',
    'q-bio'   : 'Quantitative Biology',
    'q-fin'   : 'Quantitative Finance',
    'eess'    : 'Electrical Engineering and Systems Science',
    'econ'    : 'Economics',
    'nlin'    : 'Nonlinear Sciences',
    'hep-ex'  : 'High Energy Physics - Experiment',
    'hep-lat' : 'High Energy Physics - Lattice',
    'math-ph' : 'Mathematical Physics'
}

def get_main_category(category_string):
    """
    Input  : 'cs.AI math.ST hep-th'
    Output : 'Computer Science'   (based on first category only)
    Returns None if prefix not found in category_map.
    """
    if not isinstance(category_string, str) or category_string.strip() == '':
        return None
    first_cat = category_string.strip().split(' ')[0]  # take first category
    prefix    = first_cat.split('.')[0]                 # take part before dot
    return category_map.get(prefix, None)               # None if not in map

def clean_text(text):
    """
    Light cleaning suitable for BERT.
    Removes HTML tags and collapses extra whitespace only.
    Does NOT remove punctuation or convert to lowercase.
    """
    if not isinstance(text, str) or text.strip() == '':
        return ''
    text = re.sub(r'<.*?>', '', text)   # remove HTML tags like <br> or <p>
    text = re.sub(r'\s+', ' ', text)    # collapse multiple spaces/newlines
    return text.strip()

print(f"Category map defined with {len(category_map)} categories.")
print(f"Categories: {list(category_map.values())}")



Category map defined with 20 categories.
Categories: ['Computer Science', 'Mathematics', 'Statistics', 'Astrophysics', 'High Energy Physics - Phenomenology', 'High Energy Physics - Theory', 'Quantum Physics', 'Condensed Matter', 'Nuclear Experiment', 'Nuclear Theory', 'General Relativity and Quantum Cosmology', 'Physics', 'Quantitative Biology', 'Quantitative Finance', 'Electrical Engineering and Systems Science', 'Economics', 'Nonlinear Sciences', 'High Energy Physics - Experiment', 'High Energy Physics - Lattice', 'Mathematical Physics']


In [11]:

# ================================================================
# CELL 5 — LOAD ALL CHUNKS AND COLLECT ABSTRACTS AND LABELS
#
# WHY: The arXiv dataset is too large to load into RAM all at once.
# So we use chunked reading — we read 100,000 rows at a time,
# process each chunk, collect the results into two lists, and then
# move on to the next chunk. The previous chunk is cleared from
# memory automatically when the next one loads.
#
# For each chunk we:
#   1. Keep only the 'abstract' and 'categories' columns
#   2. Drop rows where either column is empty
#   3. Clean the abstract text
#   4. Map the category code to a main category name
#   5. Drop rows where the category could not be mapped
#   6. Drop rows where the abstract is empty after cleaning
#   7. Add abstracts and labels to our running lists
#
# After this cell finishes, all_abstracts and all_labels contain
# the complete cleaned dataset ready for the next steps.
# ================================================================

print("Loading and processing all chunks...")
print("This may take several minutes depending on dataset size.")

all_abstracts = []   # will hold cleaned abstract text for every paper
all_labels    = []   # will hold category name for every paper

# create the chunk iterator
if data_file_path.endswith('.csv'):
    df_iterator = pd.read_csv(data_file_path, chunksize=chunk_size)
else:
    df_iterator = pd.read_json(data_file_path, lines=True, chunksize=chunk_size)

for i, chunk in enumerate(df_iterator):

    # Step 1: keep only the two columns we need
    chunk = chunk[['abstract', 'categories']].copy()

    # Step 2: drop rows where abstract or categories is missing
    chunk = chunk.dropna(subset=['abstract', 'categories'])

    # Step 3: clean abstract text
    chunk['abstract'] = chunk['abstract'].apply(clean_text)

    # Step 4: map categories to main category name
    chunk['main_category'] = chunk['categories'].apply(get_main_category)

    # Step 5: drop rows where category prefix not in our map
    chunk = chunk[chunk['main_category'].notna()]

    # Step 6: drop rows where abstract is empty after cleaning
    chunk = chunk[chunk['abstract'].str.strip() != '']

    # Step 7: add to our running lists
    all_abstracts.extend(chunk['abstract'].tolist())
    all_labels.extend(chunk['main_category'].tolist())

    print(f"Chunk {i+1} processed — total papers so far: {len(all_abstracts)}")

print(f"\nAll chunks processed successfully.")
print(f"Total abstracts collected : {len(all_abstracts)}")
print(f"Total labels collected    : {len(all_labels)}")

Loading and processing all chunks...
This may take several minutes depending on dataset size.
Chunk 1 processed — total papers so far: 100000
Chunk 2 processed — total papers so far: 200000
Chunk 3 processed — total papers so far: 300000
Chunk 4 processed — total papers so far: 400000
Chunk 5 processed — total papers so far: 500000
Chunk 6 processed — total papers so far: 600000
Chunk 7 processed — total papers so far: 700000
Chunk 8 processed — total papers so far: 800000
Chunk 9 processed — total papers so far: 900000
Chunk 10 processed — total papers so far: 1000000
Chunk 11 processed — total papers so far: 1100000
Chunk 12 processed — total papers so far: 1200000
Chunk 13 processed — total papers so far: 1300000
Chunk 14 processed — total papers so far: 1400000
Chunk 15 processed — total papers so far: 1500000
Chunk 16 processed — total papers so far: 1600000
Chunk 17 processed — total papers so far: 1700000
Chunk 18 processed — total papers so far: 1800000
Chunk 19 processed — tot

In [12]:

# ================================================================
# CELL 6 — LABEL ENCODING
#
# WHY: Our model cannot work with text labels like 'Computer
# Science' or 'Mathematics'. It only understands numbers.
# LabelEncoder converts each unique category name into a unique
# integer. For example:
#   'Astrophysics'       → 0
#   'Computer Science'   → 1
#   'Mathematics'        → 2  ... and so on
#
# We save the label_encoder to disk so that later, when the model
# predicts a number, we can convert it back to the category name.
# This is called inverse_transform and we use it in the prediction
# function at the end of this file.
# ================================================================

print("Encoding category labels into numbers...")

label_encoder  = LabelEncoder()
encoded_labels = label_encoder.fit_transform(all_labels)
num_classes    = len(label_encoder.classes_)

print(f"Number of unique categories : {num_classes}")
print(f"Category list: {list(label_encoder.classes_)}")

# save label encoder — needed later to decode predicted numbers
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("LabelEncoder saved as label_encoder.pkl")



Encoding category labels into numbers...
Number of unique categories : 20
Category list: ['Astrophysics', 'Computer Science', 'Condensed Matter', 'Economics', 'Electrical Engineering and Systems Science', 'General Relativity and Quantum Cosmology', 'High Energy Physics - Experiment', 'High Energy Physics - Lattice', 'High Energy Physics - Phenomenology', 'High Energy Physics - Theory', 'Mathematical Physics', 'Mathematics', 'Nonlinear Sciences', 'Nuclear Experiment', 'Nuclear Theory', 'Physics', 'Quantitative Biology', 'Quantitative Finance', 'Quantum Physics', 'Statistics']
LabelEncoder saved as label_encoder.pkl


In [13]:

# ================================================================
# CELL 7 — BUILD THE DATAFRAME
#
# WHY: The Hugging Face Dataset class (used by the Trainer) expects
# data in a specific format — a pandas DataFrame or dictionary with
# column names 'text' and 'label'. We build that DataFrame here
# from our two lists.
#
# After building the DataFrame we delete the raw lists from memory
# since we no longer need them and they take up significant RAM.
# The DataFrame is more memory efficient and is all we need going
# forward.
#
# NOTE: We do NOT sample down to 4000 rows here. We use the full
# dataset for proper training. You can uncomment the sample line
# if you want a quick test run first.
# ================================================================

print("Building DataFrame from collected abstracts and labels...")

df = pd.DataFrame({
    'text' : all_abstracts,
    'label': encoded_labels.tolist()
})

# free memory — these large lists are no longer needed
del all_abstracts
del all_labels
del encoded_labels

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head(3))

df = df.sample(100_000, random_state=42).reset_index(drop=True)
print(f"Sampled 100,000 rows for training. DataFrame shape: {df.shape}")


Building DataFrame from collected abstracts and labels...
DataFrame shape: (2993961, 2)
Columns: ['text', 'label']
                                                text  label
0  A fully differential calculation in perturbati...      8
1  We describe a new algorithm, the $(k,\ell)$-pe...     11
2  The evolution of Earth-Moon system is describe...     15
Sampled 100,000 rows for training. DataFrame shape: (100000, 2)


In [14]:

# ================================================================
# CELL 8 — CONVERT TO HUGGING FACE DATASET FORMAT
#
# WHY: The Hugging Face Trainer expects data in its own Dataset
# format, not a plain pandas DataFrame. Dataset.from_pandas()
# converts our DataFrame into that format automatically.
#
# We then split into 90% training and 10% testing using
# train_test_split. This is the standard split for large datasets.
# ================================================================

print("Converting DataFrame to Hugging Face Dataset format...")

# reset index — important step before converting to Dataset
df = df.reset_index(drop=True)

# convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# split into train and test (90% train, 10% test)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Training samples : {len(dataset['train'])}")
print(f"Testing samples  : {len(dataset['test'])}")



Converting DataFrame to Hugging Face Dataset format...
Training samples : 90000
Testing samples  : 10000


In [15]:

# ================================================================
# CELL 9 — LOAD DISTILBERT TOKENIZER AND TOKENIZE DATA
#
# WHY: DistilBERT does not understand raw text. It needs text
# converted into numerical tokens first. The tokenizer does this.
#
# distilbert-base-uncased means:
#   - 'distilbert' → smaller, faster version of BERT (40% smaller,
#      twice as fast, 97% of BERT accuracy)
#   - 'base'       → standard size (not large or small variant)
#   - 'uncased'    → converts everything to lowercase internally
#
# tokenize function does three things for each abstract:
#   1. Converts words to token IDs (numbers DistilBERT understands)
#   2. Adds padding so all abstracts reach exactly max_length=128
#      tokens (shorter ones are padded, longer ones are cut)
#   3. Creates attention_mask — tells model which positions are
#      real text (1) and which are just padding (0)
#
# max_length=128 is a balance — 96 is faster but loses some info
# for longer abstracts. 256 is more accurate but slower to train.
# 128 works well for most research paper abstracts.
#
# batched=True processes multiple abstracts at once for speed.
# ================================================================

print("Loading DistilBERT tokenizer...")

tokenizer  = AutoTokenizer.from_pretrained("distilbert-base-uncased")
MAX_LENGTH = 128   # max number of tokens per abstract

def tokenize(batch):
    """
    Converts raw text into token IDs and attention masks.
    Called on batches of abstracts for efficiency.
    """
    return tokenizer(
        batch["text"],
        padding    = "max_length",   # pad short texts to MAX_LENGTH
        truncation = True,           # cut long texts at MAX_LENGTH
        max_length = MAX_LENGTH
    )

print(f"Tokenizing dataset with max_length={MAX_LENGTH}...")
dataset = dataset.map(tokenize, batched=True)

# tell the Dataset which columns are actual model inputs
# label must be present as a column for the Trainer to use it
dataset.set_format(
    type    = 'torch',
    columns = ['input_ids', 'attention_mask', 'label']
)

print("Tokenization complete.")
print(f"Training set columns: {dataset['train'].column_names}")


Loading DistilBERT tokenizer...


Tokenizing dataset with max_length=128...


Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenization complete.
Training set columns: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [16]:

# ================================================================
# CELL 10 — LOAD DISTILBERT MODEL WITH CLASSIFICATION HEAD
#
# WHY: We use AutoModelForSequenceClassification which loads
# DistilBERT and automatically adds a classification layer on top.
# This classification layer has num_classes output neurons — one
# for each of our 20 arXiv categories.
#
# The model has two parts:
#   1. DistilBERT base — 66 million parameters already trained on
#      a massive amount of English text. It understands language.
#   2. Classification head — a single linear layer on top that we
#      train to map DistilBERT's understanding to our 20 categories.
#
# We move the model to GPU using model.to(device) so training
# uses the fast GPU instead of the slow CPU.
# ================================================================

print(f"Loading DistilBERT model with {num_classes} output classes...")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = num_classes    # 20 categories from our arXiv data
)

model.to(device)
print(f"Model loaded and moved to: {device}")
print(f"Output classes: {num_classes}")

Loading DistilBERT model with 20 output classes...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to: cuda
Output classes: 20


In [17]:

# ================================================================
# CELL 11 — DEFINE COMPUTE METRICS FUNCTION
#
# WHY: The Trainer needs a function that takes predictions and
# true labels, and returns the metrics we care about. We compute
# accuracy and weighted F1 score after each evaluation step.
#
# accuracy   → percentage of papers classified correctly overall
# f1 score   → balanced measure of precision and recall across
#               all 20 categories (weighted by category size)
# ================================================================

def compute_metrics(eval_pred):
    """
    Called by Trainer after each evaluation.
    Returns accuracy and weighted F1 score.
    """
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)   # take highest scoring class

    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='weighted')

    return {
        'accuracy': round(acc, 4),
        'f1'      : round(f1, 4)
    }

print("compute_metrics function defined.")



compute_metrics function defined.


In [18]:


# ================================================================
# CELL 12 — SET TRAINING ARGUMENTS
#
# WHY: TrainingArguments controls every aspect of how training
# runs. Each setting is explained below.
#
# output_dir                  → where to save checkpoints
# per_device_train_batch_size → 16 papers per GPU step (fits in
#                               GPU memory with MAX_LENGTH=128)
# per_device_eval_batch_size  → same batch size for evaluation
# fp16                        → half precision (16-bit) training
#                               on GPU — makes training ~2x faster
#                               automatically disabled on CPU
# num_train_epochs            → 3 full passes through training data.
#                               BERT needs fewer epochs than plain
#                               neural networks — 3-4 is enough
# evaluation_strategy         → evaluate on test set after each
#                               epoch so we can see progress
# save_strategy               → save model checkpoint each epoch
# load_best_model_at_end      → after all epochs, load the epoch
#                               that had the best test accuracy
# logging_steps               → print training loss every 100 steps
# learning_rate               → 2e-5 is the standard rate for
#                               fine-tuning BERT. Too high = unstable
#                               training. Too low = very slow learning
# warmup_ratio                → gradually increase learning rate for
#                               first 6% of steps — helps stability
# weight_decay                → small regularization to prevent
#                               the model memorizing training data
# dataloader_num_workers      → parallel data loading threads
# report_to                   → 'none' disables wandb/tensorboard
# ================================================================

training_args = TrainingArguments(
    output_dir                  = "./results",
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    fp16                        = torch.cuda.is_available(),
    num_train_epochs            = 3,
    eval_strategy               = "epoch",   # ← changed
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    logging_steps               = 100,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.06,
    weight_decay                = 0.01,
    dataloader_num_workers      = 2,
    report_to                   = "none"
)

print("TrainingArguments configured.")
print(f"Epochs         : {training_args.num_train_epochs}")
print(f"Batch size     : {training_args.per_device_train_batch_size}")
print(f"Learning rate  : {training_args.learning_rate}")
print(f"FP16 training  : {training_args.fp16}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments configured.
Epochs         : 3
Batch size     : 16
Learning rate  : 2e-05
FP16 training  : True


In [19]:


# ================================================================
# CELL 13 — CREATE TRAINER AND START TRAINING
#
# WHY: The Hugging Face Trainer handles the entire training loop
# for us. We just pass it the model, arguments, datasets, and
# metrics function — it takes care of:
#   - Feeding batches to the model
#   - Calculating loss after each batch
#   - Running backpropagation to compute gradients
#   - Updating model weights with Adam optimizer
#   - Evaluating on test set after each epoch
#   - Saving the best model checkpoint
#   - Printing training progress
#
# trainer.train() starts the actual training. Progress is printed
# every 100 steps (logging_steps=100) showing loss and accuracy.
# After each full epoch it evaluates on the test set.
# ================================================================

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = dataset["train"],
    eval_dataset    = dataset["test"],
    compute_metrics = compute_metrics
)

print("Trainer created. Starting training...")
print(f"Total training samples : {len(dataset['train'])}")
print(f"Total testing samples  : {len(dataset['test'])}")
print("=" * 60)

trainer.train()

print("\nTraining complete!")


Trainer created. Starting training...
Total training samples : 90000
Total testing samples  : 10000


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.604458,0.613629,0.801700,0.789800
2,0.521591,0.543793,0.820000,0.813300
3,0.405441,0.561433,0.817800,0.811600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete!


In [20]:
# ================================================================
# CELL 14 — FINAL EVALUATION (no Trainer, direct model evaluation)
# ================================================================

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

print("Running final evaluation on test set...")

model.eval()

all_preds  = []
all_labels = []

dataloader = DataLoader(dataset["test"], batch_size=32)

for batch in dataloader:
    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels         = batch["label"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
    all_preds.extend(preds)
    all_labels.extend(labels.cpu().numpy())

# compute metrics
acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average='weighted')

print(f"\nFinal Test Results:")
print(f"  Accuracy : {round(acc, 4)}")
print(f"  F1 Score : {round(f1, 4)}")

print("\nClassification Report (per category):")
print(classification_report(
    all_labels,
    all_preds,
    target_names = label_encoder.classes_
))

Running final evaluation on test set...

Final Test Results:
  Accuracy : 0.82
  F1 Score : 0.8133

Classification Report (per category):
                                            precision    recall  f1-score   support

                              Astrophysics       0.95      0.92      0.94      1085
                          Computer Science       0.87      0.91      0.89      2457
                          Condensed Matter       0.85      0.87      0.86      1205
                                 Economics       0.50      0.39      0.44        33
Electrical Engineering and Systems Science       0.53      0.30      0.39       227
  General Relativity and Quantum Cosmology       0.65      0.76      0.70       234
          High Energy Physics - Experiment       0.80      0.73      0.76        89
             High Energy Physics - Lattice       0.68      0.65      0.66        68
       High Energy Physics - Phenomenology       0.79      0.87      0.83       510
              High En

In [21]:

# ================================================================
# CELL 17 — PREDICT CATEGORY OF A NEW RESEARCH PAPER
#
# WHY: This is the final goal of the entire project — given any
# new research paper abstract, predict which academic category
# it belongs to.
#
# predict_category function does four steps:
#   1. Clean the abstract text (same cleaning as training)
#   2. Tokenize it using the same tokenizer and settings
#   3. Pass it through the trained model to get 20 scores
#   4. Pick the category with the highest score and decode it
#      back to a readable name using label_encoder
# ================================================================

def predict_category(abstract_text):
    """
    Input : any research paper abstract as a string
    Output: predicted category name as a string

    Example:
        predict_category("This paper proposes a new neural network...")
        → 'Computer Science'
    """
    model.eval()   # switch to evaluation mode (turns off dropout)

    # Step 1: clean text
    cleaned = clean_text(abstract_text)

    # Step 2: tokenize
    inputs = tokenizer(
        cleaned,
        return_tensors = 'pt',
        padding        = 'max_length',
        truncation     = True,
        max_length     = MAX_LENGTH
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Step 3: get model prediction
    with torch.no_grad():   # no gradient calculation needed for prediction
        outputs  = model(**inputs)
        pred_idx = torch.argmax(outputs.logits, dim=1).item()

    # Step 4: decode number back to category name
    category = label_encoder.inverse_transform([pred_idx])[0]
    return category


In [22]:
# ================================================================
# TEST YOUR MODEL — paste any abstract below
# ================================================================

my_abstract = """
In-hospital mortality is one of the most critical indicators of surgical care quality and patient safety. Accurate prediction of mortality risk before and during surgery enables clinicians to make timely decisions, allocate resources efficiently, and reduce preventable complications. This study investigates the predictive power of both classical statistical models and modern machine learning techniques by incorporating a wide range of preoperative and intraoperative risk factors.
A retrospective dataset of surgical patients was analysed, containing demographic characteristics, comorbidities, laboratory values, intraoperative hemodynamic trends, blood loss, and operative duration. Classical modelling approaches such as Logistic Regression and Cox Regression were developed and compared with advanced machine learning algorithms including Random Forest, Boost, and fine-tuned Transformer-based classifiers. Model performance was evaluated using standard metrics such as Accuracy, AUC, F1-score, Brier Score, and calibration plots. Explainability and feature importance were assessed using SHAP values to ensure clinical interpretability.
Findings show that machine learning models—particularly Boost and Transformer-based architectures—achieved superior predictive performance, identifying complex nonlinear interactions between risk factors that traditional models could not fully capture. Classical statistical models, however, demonstrated higher interpretability and remained stable across smaller subgroups of patients. Key predictors of mortality included preoperative comorbidity burden, intraoperative blood pressure variability, operative duration, and metabolic markers.
Overall, the study concludes that integrating both classical and machine learning approaches provides a more comprehensive framework for mortality risk prediction. While ML models deliver stronger predictive accuracy, classical models contribute essential interpretability required for clinical decision-making. Combining these methodologies can support hospitals in building reliable early-warning systems, improving patient outcomes, and strengthening perioperative risk management.

"""

predicted = predict_category(my_abstract)
print(f"Predicted Category: {predicted}")

Predicted Category: Statistics


In [23]:
my_abstract = """
The study conducted a statistical analysis of career choices among youth in a specific locality, aiming to identify patterns, preferences, and influencing factors. The findings highlight a strong preference for careers in technology and healthcare, driven by the growing demand in these industries. The study also emphasizes the significant impact of socio-economic background on career decisions. Youth from higher-income families are more inclined to pursue higher education and professional careers, while those from lower-income backgrounds often opt for vocational training and immediate employment. This suggests that economic factors heavily influence career paths, with wealthier individuals having more opportunities to access advanced education and professional careers, whereas those from less affluent backgrounds prioritize financial stability and quicker entry into the workforce. The study's insights are crucial for policymakers and educators in understanding and addressing the disparities in career opportunities and aspirations among different socio-economic groups
"""

predicted = predict_category(my_abstract)
print(f"Predicted Category: {predicted}")

Predicted Category: Economics


In [24]:
# ================================================================
# SAVE MODEL TO DISK
# ================================================================

import os
import pickle

# this will create a 'model' folder right where your notebook is
save_path = "model/"
os.makedirs(save_path, exist_ok=True)

# save model weights
model.save_pretrained(save_path)
print("model weights saved")

# save tokenizer
tokenizer.save_pretrained(save_path)
print("tokenizer saved")

# save label encoder
with open(os.path.join(save_path, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)
print("label encoder saved")

# save category map
with open(os.path.join(save_path, "category_map.pkl"), "wb") as f:
    pickle.dump(category_map, f)
print("category map saved")

# show all saved files
print("\nAll files saved successfully:")
for file in os.listdir(save_path):
    size = os.path.getsize(os.path.join(save_path, file))
    print(f"  {file:40s}  {round(size/1024/1024, 2)} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model weights saved
tokenizer saved
label encoder saved
category map saved

All files saved successfully:
  category_map.pkl                          0.0 MB
  config.json                               0.0 MB
  label_encoder.pkl                         0.0 MB
  model.safetensors                         255.48 MB
  tokenizer.json                            0.68 MB
  tokenizer_config.json                     0.0 MB


In [25]:
import os

save_path = "model/"

print("Files in model/ folder:")
for file in os.listdir(save_path):
    size = os.path.getsize(os.path.join(save_path, file))
    print(f"  {file:40s}  {round(size/1024/1024, 2)} MB")

Files in model/ folder:
  category_map.pkl                          0.0 MB
  config.json                               0.0 MB
  label_encoder.pkl                         0.0 MB
  model.safetensors                         255.48 MB
  tokenizer.json                            0.68 MB
  tokenizer_config.json                     0.0 MB


In [26]:
import pickle
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

save_path     = "model/"
tokenizer     = AutoTokenizer.from_pretrained(save_path)
model         = AutoModelForSequenceClassification.from_pretrained(save_path)
label_encoder = pickle.load(open(save_path + "label_encoder.pkl", "rb"))
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

MAX_LENGTH = 128

def clean_text(text):
    import re
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def predict_category(abstract_text):
    cleaned = clean_text(abstract_text)
    inputs  = tokenizer(
        cleaned,
        return_tensors = "pt",
        padding        = "max_length",
        truncation     = True,
        max_length     = MAX_LENGTH
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs  = model(**inputs)
        pred_idx = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([pred_idx])[0]

print("Model loaded successfully!")
print(f"Device     : {device}")
print(f"Categories : {list(label_encoder.classes_)}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!
Device     : cuda
Categories : ['Astrophysics', 'Computer Science', 'Condensed Matter', 'Economics', 'Electrical Engineering and Systems Science', 'General Relativity and Quantum Cosmology', 'High Energy Physics - Experiment', 'High Energy Physics - Lattice', 'High Energy Physics - Phenomenology', 'High Energy Physics - Theory', 'Mathematical Physics', 'Mathematics', 'Nonlinear Sciences', 'Nuclear Experiment', 'Nuclear Theory', 'Physics', 'Quantitative Biology', 'Quantitative Finance', 'Quantum Physics', 'Statistics']


In [27]:
my_abstract = """
We propose a new deep learning model using transformer architecture
for natural language processing tasks achieving state of the art
results on text classification benchmarks.
"""

print(f"Predicted: {predict_category(my_abstract)}")

Predicted: Computer Science
